# Python Master Notes — OOP Wrap-up and Functional Python Tools
**Class date: 25 October 2026**

This notebook organizes the supplied class notebook and recording transcript into one revision-friendly resource. It follows the class material:

1. Static/class variables and static methods
2. Special methods (dunder methods) and operator overloading
3. Getters, setters and the `@property` decorator
4. Lambda functions
5. `map()` and `filter()`
6. Unpacking and merging collections
7. List and dictionary comprehensions
8. `enumerate()`
9. Generators and `yield`
10. `zip()`
11. `match-case`
12. The instructor's closing guidance on planning small projects with AI

The supplied PDF had no extractable text layer. The notebook and transcript are therefore the sources for these notes. Examples are kept close to the class approach, with technical clarifications explicitly identified.

## How to use this notebook
- Run the code cells from top to bottom.
- Some examples define classes or functions that are reused later.
- Interactive `input()` examples are shown as optional commented code so the notebook does not pause unexpectedly.
- For package/file-project practice, create the named `.py` files in a normal project folder; a notebook cell is not a substitute for a multi-file project.
- Try the dry-run questions before reading their explanations.

# Part 1 — Static (class) variables

## 1.1 What is a static variable?
The class notebook calls these **static variables**. In Python, the usual term is **class variable**.

A class variable is defined directly inside the class body, outside methods. It belongs to the class and is shared as the default value by instances that do not shadow it with an instance attribute.

In the class example:
- `course_name` is a class variable.
- `mentor` is another class variable.
- `batch_number` is an instance variable because it is assigned through `self` inside `__init__`.

In [ ]:
class User:
    course_name = "UDS"                 # class variable
    mentor = ["Monal", "Paul sir"]      # class variable

    def __init__(self, batch_no):
        self.batch_number = batch_no    # instance variable

    def status_value_get(self):
        print(self.course_name)
        print(User.course_name)

user_1 = User(3.0)
user_2 = User(3.0)

print(user_1.batch_number)
print(user_2.batch_number)
print(User.course_name)
print(User.mentor)

## 1.2 Updating a class variable
The class example updates the class variable using the class name:

In [ ]:
User.course_name = "Ultimate Data Science"

user_a = User(3.0)
user_b = User(4.0)

print(user_a.course_name)
print(user_b.course_name)
print(User.course_name)

When an instance looks up `course_name`, Python first checks the instance and then the class. If the instance does not have its own attribute with that name, it finds the class variable.

**Important distinction:** assigning through an instance does not update the shared class variable. It creates or updates an instance attribute that shadows the class variable for that instance.

In [ ]:
user_a.course_name = "FDE"  # creates an instance attribute; does not change User.course_name

print(user_a.course_name)   # FDE
print(user_b.course_name)   # Ultimate Data Science
print(User.course_name)     # Ultimate Data Science

### Dry run: where does Python find the value?
For `user_a.course_name`:
1. Python checks `user_a`'s instance attributes.
2. It finds `"FDE"` and returns it.
3. It does not need to look up `User.course_name`.

For `user_b.course_name`:
1. Python checks `user_b`'s instance attributes.
2. No `course_name` is found.
3. Python checks the class and finds `"Ultimate Data Science"`.

The class notebook's comment says not to update a static value through an object reference. The practical reason is that it can create a separate instance attribute rather than updating the shared class value.

## 1.3 Mutable class variables — a caution
A mutable class variable such as a list is shared by instances unless an instance shadows it. Mutating that shared list through one instance affects what other instances see.

In [ ]:
class Team:
    members = []

team_a = Team()
team_b = Team()

team_a.members.append("Alex")

print(team_a.members)
print(team_b.members)  # same shared list

If each object needs its own list, create it in `__init__` using `self.members = []`. Use class variables for data genuinely shared across all instances.

In [ ]:
class SeparateTeams:
    def __init__(self):
        self.members = []

team_x = SeparateTeams()
team_y = SeparateTeams()

team_x.members.append("Alex")

print(team_x.members)
print(team_y.members)

# Part 2 — Static methods

## 2.1 What is a static method?
A static method is a function placed inside a class namespace and marked with `@staticmethod`.

- It does not receive an instance automatically as `self`.
- It does not receive the class automatically as `cls`.
- It can be called using the class name, without creating an object.
- It can also be called through an instance, although the class-name form makes its independence from instance state clearer.

The class example uses `MathUtils` to group simple arithmetic operations.

In [ ]:
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b

    @staticmethod
    def subtract_a_from_b(a, b):
        return a - b

    @staticmethod
    def subtract_b_from_a(a, b):
        return b - a

    def class_method(self):
        print(self.add(1, 2))
        print(MathUtils.add(1, 2))

print(MathUtils.add(1, 2))

obj_1 = MathUtils()
obj_1.class_method()

### Static method vs instance method vs class method

| Method type | Decorator | First automatic parameter | Typical access |
|---|---|---|---|
| Instance method | None | `self` (the instance) | Object |
| Class method | `@classmethod` | `cls` (the class) | Class or object |
| Static method | `@staticmethod` | None | Usually class |

Only static and instance methods were demonstrated in this class notebook. `@classmethod` is included in the comparison to distinguish the three common method types.

# Part 3 — Special methods (dunder methods)

## 3.1 What are special methods?
Special methods are methods with names surrounded by double underscores, such as `__init__`, `__str__`, `__repr__`, `__add__`, `__len__` and `__getitem__`.

They let a custom class define how objects behave with built-in operations and syntax.

The class notes mention:
- `__new__` — creates and returns a new instance.
- `__init__` — initializes the already-created instance.

A useful technical clarification: `__new__` is the object-creation method; `__init__` initializes the object after it has been created. In ordinary classes, you usually customize `__init__`, not `__new__`.

## 3.2 `__str__` and `__repr__`
The `Car` example defines two string representations:
- `__str__`: a readable representation intended for users.
- `__repr__`: a more detailed, unambiguous representation intended for developers and debugging.

In [ ]:
class Car:
    def __init__(self, manuf, car_name):
        self.manuf = manuf
        self.car_name = car_name

    def __str__(self):
        return f"Manufacturer Name: {self.manuf} and we produce car."

    def __repr__(self):
        return (
            f"Class Car, Manufacturer Name: {self.manuf}. "
            f"This object contains the car model: {self.car_name}"
        )

car_1 = Car("toyota", "glanza")

print(str(car_1))
print(repr(car_1))
print(f"{car_1!s}")  # uses __str__
print(f"{car_1!r}")  # uses __repr__

The format specifiers `!s` and `!r` request string conversion and representation conversion respectively. The class also demonstrated formatting a normal floating-point number to two decimal places:

In [ ]:
variable_num = 2232.334332
print(f"{variable_num:.2f}")

## 3.3 `__add__`, `__sub__` and `__eq__`: operator overloading
The `Wallet` example makes `+`, `-` and `==` work with objects of the custom class.

- `wallet_a + wallet_b` calls `wallet_a.__add__(wallet_b)`.
- `wallet_a - wallet_b` calls `wallet_a.__sub__(wallet_b)`.
- `wallet_a == wallet_b` calls the equality comparison implementation, here `__eq__`.

The class returns a new `Wallet` object from addition and subtraction, and a Boolean from equality.

In [ ]:
class Wallet:
    def __init__(self, cash):
        self.cash = cash

    def __repr__(self):
        return f"Wallet(cash={self.cash})"

    def __add__(self, other):
        return Wallet(self.cash + other.cash)

    def __sub__(self, other):
        return Wallet(self.cash - other.cash)

    def __eq__(self, other):
        if not isinstance(other, Wallet):
            return NotImplemented
        return self.cash == other.cash

w_obj_1 = Wallet(100)
w_obj_2 = Wallet(200)
w_obj_3 = Wallet(200)

combined_sum_obj = w_obj_1 + w_obj_2 + w_obj_3

print(combined_sum_obj)
print(combined_sum_obj.cash)
print(w_obj_2 == w_obj_3)
print(w_obj_1 == w_obj_2)

### Dry run: why can three wallets be added?
For `w_obj_1 + w_obj_2 + w_obj_3`, Python evaluates the operations from left to right:
1. `w_obj_1 + w_obj_2` calls `__add__` and returns a new `Wallet(300)`.
2. The intermediate result is still a `Wallet` object.
3. That new wallet can be added to `w_obj_3`.
4. The second `__add__` returns `Wallet(500)`.

This works because the result of the first addition has the same type and supports the next addition.

**Practical note from the instructor:** custom operator behaviour is useful to understand, but the class discussion emphasized knowing what the special methods do rather than spending excessive time inventing custom data structures for everyday data-science work.

## 3.4 `__len__` and `__getitem__`: make an object support familiar operations
The `ATM` example stores transaction values in a list. It then defines:
- `__len__` so `len(atm_object)` returns the number of transactions.
- `__getitem__` so `atm_object[index]` returns a transaction at that index.

In [ ]:
class ATM:
    def __init__(self):
        self.transaction_history = []

    def __repr__(self):
        return "Class ATM"

    def do_transaction(self, value):
        self.transaction_history.append(value)
        print(f"Transaction done: {value}")

    def __len__(self):
        return len(self.transaction_history)

    def __getitem__(self, index):
        return self.transaction_history[index]

atm_obj_1 = ATM()
atm_obj_1.do_transaction(-100)
atm_obj_1.do_transaction(-1000)
atm_obj_1.do_transaction(10000)

print(len(atm_obj_1.transaction_history))
print(len(atm_obj_1))
print(atm_obj_1[1])
print(atm_obj_1[0])

The class notebook also calculates an example fee using the number of transactions:

In [ ]:
transaction_fee = len(atm_obj_1) * 0.1
print(transaction_fee)

The object is not literally a list. The developer has chosen to support selected list-like operations. The user can only perform the behaviours the class implements.

### Special methods covered in this class

| Special method | Operation or built-in that uses it | Class example |
|---|---|---|
| `__init__` | Object initialization | `Car`, `Wallet`, `ATM` |
| `__str__` | `str(obj)`, readable display | `Car` |
| `__repr__` | `repr(obj)`, developer representation | `Car`, `Wallet`, `ATM` |
| `__add__` | `obj1 + obj2` | `Wallet` |
| `__sub__` | `obj1 - obj2` | `Wallet` |
| `__eq__` | `obj1 == obj2` | `Wallet` |
| `__len__` | `len(obj)` | `ATM` |
| `__getitem__` | `obj[index]` | `ATM` |
| `__new__` | Creates an instance | Mentioned conceptually |

# Part 4 — Getters and setters

## 4.1 What are getter and setter methods?
A **getter** is a method used to retrieve an attribute's value. A **setter** is a method used to update an attribute's value.

The class uses a private-style attribute `__balance` and provides controlled methods to read and update it. This lets the class decide which values it exposes and what validation it performs.

In [ ]:
class Account:
    def __init__(self, balance):
        self.__balance = balance

    def get_balance(self):
        return self.__balance

    def set_balance(self, value):
        self.__balance = value

account_1 = Account(1000)

print(account_1.get_balance())
account_1.set_balance(100000)
print(account_1.get_balance())

This is the traditional getter/setter approach shown in the class:
- `get_balance()` reads the value.
- `set_balance(value)` changes the value.

The example setter does not validate the new value. Validation can be added inside the setter.

## 4.2 Property decorator: getter
The second approach uses `@property`. It allows a method to be accessed using attribute syntax.

In [ ]:
class AccountV2:
    def __init__(self, balance):
        self.__balance = balance

    @property
    def balance(self):
        print("Checking balance")
        return self.__balance

account_2 = AccountV2(1000)
print(account_2.balance)

`account_2.balance` looks like an attribute access, but Python calls the getter method behind the scenes.

## 4.3 Property setter
The setter is attached to the property using `@balance.setter`. The method uses the same name as the property.

In [ ]:
class AccountV2:
    def __init__(self, balance):
        self.__balance = balance

    @property
    def balance(self):
        print("Checking balance")
        return self.__balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            print("Security Alert: balance cannot be negative")
        elif value > 1_000_000:
            print("Security Alert: large transaction requires approval")
        else:
            print("Changing balance")
            self.__balance = value

account_2 = AccountV2(1000)

print(account_2.balance)
account_2.balance = -899
print(account_2.balance)
account_2.balance = 5000
print(account_2.balance)

Notice the syntax:
- Read: `account_2.balance`
- Update: `account_2.balance = 5000`

The setter validates the new value before changing the private-style attribute.

## 4.4 Property deleter
The class notebook also defines a deleter using `@balance.deleter`. In the supplied example, it prints a warning rather than deleting the stored balance.

In [ ]:
class AccountV3:
    def __init__(self, balance):
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            print("Security Alert: balance cannot be negative")
        elif value > 1_000_000:
            print("Security Alert: large transaction requires approval")
        else:
            self.__balance = value

    @balance.deleter
    def balance(self):
        print("ERROR: You cannot delete this balance through the property")

account_3 = AccountV3(1000)
print(account_3.balance)
del account_3.balance
print(account_3.balance)

### Getter/setter/deleter syntax summary

| Purpose | Decorator / syntax | Example |
|---|---|---|
| Getter | `@property` | `value = account.balance` |
| Setter | `@balance.setter` | `account.balance = 500` |
| Deleter | `@balance.deleter` | `del account.balance` |

The class explains these as three roles: getting, setting and deleting. A property provides a controlled interface while keeping attribute-style syntax.

# Part 5 — Lambda functions

## 5.1 What is a lambda?
A lambda is a small anonymous function written as a single expression. The class describes it as a one-line function.

A normal function:

In [ ]:
def add(a, b):
    return a + b

answer_1 = add(1, 2)
print(answer_1)

Equivalent lambda expression:

In [ ]:
add_lambda = lambda a, b: a + b

print(add_lambda(1, 2))

cube = lambda x: x ** 3
print(cube(2))

adult_check = lambda age: True if age > 18 else False
print(adult_check(10))
print(adult_check(19))

A lambda expression contains:
- `lambda`
- its parameters
- a colon
- one expression whose result is returned automatically

Use `def` for functions that need multiple statements, documentation, complex logic or a meaningful reusable name. The class focuses on simple lambda examples.

# Part 6 — `map()`

## 6.1 What does `map()` do?
`map(function, iterable)` applies a function to each item from an iterable and returns a map iterator. Convert it to a list when you want to display or store all results at once.

In [ ]:
def square_of_number(x):
    return x * x

list_a = [1, 2, 3, 4, 5]

list_answer = []
for i in list_a:
    list_answer.append(square_of_number(i))

print(list_answer)

mapped_answer = list(map(square_of_number, list_a))
print(mapped_answer)

The `map()` version performs the same transformation as the explicit loop:
- takes one item,
- passes it to `square_of_number`,
- collects the returned result in the map iterator.

## 6.2 `map()` with tuples and multiple values
The class also passes each tuple to a function that accepts one tuple and adds its first two elements.

In [ ]:
def add_pair(values):
    return values[0] + values[1]

list_b = [(1, 2), (3, 4), (100, 3)]

print(list(map(add_pair, list_b)))

## 6.3 `map()` with a lambda

In [ ]:
ages = [34, 45, 10, 19, 4]
adult_results = list(map(adult_check, ages))
print(adult_results)

**Remember:** `map()` transforms every item. It does not select only some items based on a condition; that is the role of `filter()`.

# Part 7 — `filter()`

## 7.1 What does `filter()` do?
`filter(function, iterable)` keeps only the items for which the function returns a truthy result. In the class definition, the filtering function takes one item and returns a Boolean.

In [ ]:
list_a = [1, 2, 3, 4, 5]

def is_even(num):
    return num % 2 == 0

even_numbers = list(filter(is_even, list_a))
print(even_numbers)

### `map()` vs `filter()`

| Function | Main purpose | Example |
|---|---|---|
| `map()` | Transform each item | `[1, 2, 3]` → `[1, 4, 9]` |
| `filter()` | Keep items meeting a condition | `[1, 2, 3, 4]` → `[2, 4]` |

Both return iterators in Python 3. Use `list(...)` to materialize their results.

# Part 8 — Unpacking and merging collections

## 8.1 Basic unpacking
Unpacking assigns the elements of an iterable to individual variables. The number of target variables must match the number of values unless starred unpacking is used.

In [ ]:
list_c = [1, 2, 3, 4, 5]

a, b, c, d, e = list_c
print(a, b, c, d, e)

This raises a `ValueError` because there are five values but only three target variables:

In [ ]:
# Uncomment to see the error:
# a, b, c = list_c

## 8.2 Starred unpacking
A starred target collects the remaining values into a list.

In [ ]:
a, b, *c = list_c
print(a, b, c)

first, *middle, last = list_c
print(first, middle, last)

## 8.3 Merging lists
The class shows three ideas: nesting lists, concatenating them with `+`, and unpacking their elements into a new list.

In [ ]:
list_a = [1, 2]
list_b = [3, 4]

nested_list = [list_a, list_b]
print(nested_list)

merged_with_plus = list_a + list_b
print(merged_with_plus)

merged_with_unpacking = [*list_a, *list_b]
print(merged_with_unpacking)

`[list_a, list_b]` creates a list containing two inner lists. It does not flatten them. The `+` and starred-unpacking versions create one flat list containing the values.

## 8.4 Unpacking when calling a function
The `*` operator can unpack a tuple or list into positional arguments.

In [ ]:
def add(a, b):
    return a + b

pairs = [(1, 2), (2, 3), (1, 4)]

for pair in pairs:
    print(add(*pair))

For each loop iteration, `add(*pair)` unpacks the two tuple values into `a` and `b`.

# Part 9 — List and dictionary comprehensions

## 9.1 List comprehension
The class first builds a list of even values using a normal loop, then introduces list comprehension as a shorter way to build a new list.

In [ ]:
numbers = list(range(1, 21))
print(numbers)

def even_numbers_loop(values):
    final_even_list = []
    for i in values:
        if i % 2 == 0:
            final_even_list.append(i)
    return final_even_list

print(even_numbers_loop(numbers))

General patterns:
```python
[expression for item in iterable]
[expression for item in iterable if condition]
[expression_if_true if condition else expression_if_false for item in iterable]
```

In [ ]:
# Add 1 to every number
plus_one = [num + 1 for num in numbers]
print(plus_one)

# Keep even numbers only
evens = [num for num in numbers if num % 2 == 0]
print(evens)

# Keep the number if even; otherwise place None
even_or_none = [num if num % 2 == 0 else None for num in numbers]
print(even_or_none)

**Order matters:** when filtering, the `if condition` appears after the `for` clause. When choosing between two output expressions, the `if ... else ...` expression appears before the `for` clause.

## 9.2 Dictionary comprehension
A dictionary comprehension builds key-value pairs using an expression and a loop.

In [ ]:
even_status = {num: (num % 2 == 0) for num in range(1, 101)}

print(type(even_status))
print(even_status)
print(even_status[4])
print(even_status[5])

In this example:
- each number from 1 through 100 is a dictionary key,
- the value is `True` when the number is even,
- otherwise the value is `False`.

# Part 10 — `enumerate()`

## 10.1 Why use `enumerate()`?
The class first tracks a counter manually, then replaces that pattern with `enumerate()`. `enumerate(iterable, start=...)` yields pairs containing an index/counter and the corresponding value.

In [ ]:
values = list(range(1, 17, 3))
print(values)

count = 1
for value in values:
    print(f"value from list: {value}, count: {count}")
    count += 1

print("--- using enumerate ---")
for count, value in enumerate(values, start=1):
    print(f"value from list: {value}, count: {count}")

Use `_` when you intentionally do not need one of the returned values. This communicates that the value is being ignored.

In [ ]:
for _, value in enumerate(range(1, 10)):
    print(value)

# Part 11 — Iterators and generators

## 11.1 Iterator concept
The transcript explains generators in relation to iterators:
- An iterator is an object that produces values one at a time.
- The two key methods for an iterator are `__iter__()` and `__next__()`.
- A generator is a simpler way to create an iterator using a function and `yield`.

**Relationship:** every generator is an iterator, but not every iterator is a generator. A list iterator, for example, is an iterator but is not a generator function's result.

## 11.2 Generator function and `yield`
A function containing `yield` is a generator function. Calling it returns a generator object. The function body runs as values are requested.

In [ ]:
def get_even_number(values):
    for value in values:
        if value % 2 == 0:
            yield value

list_d = range(1, 20)
generator_a = get_even_number(list_d)

print(type(generator_a))
print(next(generator_a))
print("Other work can run between requests.")
print(next(generator_a))
print(next(generator_a))

### What happens when `next()` is called?
1. The generator starts running, or resumes from where it paused.
2. It checks values in the loop.
3. When it reaches `yield value`, it gives that value to the caller and pauses.
4. The next `next(generator_a)` resumes after the previous `yield`.
5. When no more values remain, the generator raises `StopIteration`.

The transcript emphasizes that values are produced when requested, rather than building a complete result list up front.

## 11.3 Generator vs returning a complete list
The class notebook compares yielding one even number at a time with collecting all even numbers into a list and yielding that list once.

In [ ]:
def get_even_number_one_list(values):
    collected = []
    for value in values:
        if value % 2 == 0:
            collected.append(value)
    yield collected

generator_b = get_even_number_one_list(range(1, 20))

print(next(generator_b))

This second function yields once, after the loop finishes. After that one yield, the generator is exhausted. Calling `next()` again raises `StopIteration`.

In [ ]:
generator_c = get_even_number_one_list(range(1, 10))

try:
    print(next(generator_c))
    print(next(generator_c))
except StopIteration:
    print("The generator has completed.")

## 11.4 Handling generator completion
A generator signals that it has no more values by raising `StopIteration`. A `for` loop handles this automatically. If calling `next()` manually, you can catch `StopIteration`.

In [ ]:
generator_d = get_even_number(range(1, 8))

while True:
    try:
        value = next(generator_d)
        print(value)
    except StopIteration:
        print("No more values.")
        break

### Generator advantages and limits
- Produces values lazily, only when requested.
- Can avoid building a large result list in memory.
- Remembers its execution state between `next()` calls.
- Once exhausted, a normal generator does not restart; create a new generator object to iterate again.

The transcript mentions concurrency and asynchronous programming as contexts where producing work incrementally can be useful, but those topics were explicitly outside this class's scope.

# Part 12 — `zip()`

## 12.1 The problem: related values stored in separate iterables
The class uses two lists: names and scores. The goal is to process each person's name and corresponding score together.

In [ ]:
names = ["Shravan", "Narsing", "Alex", "Anushka"]
scores = [85, 78, 90]

# Index-based approach; the lists do not have equal lengths.
for i in range(len(names)):
    try:
        print(f"{names[i]} scored {scores[i]}")
    except IndexError:
        pass

The index-based approach needs extra care when the lists have different lengths. The class then introduces `zip()` to pair corresponding items.

In [ ]:
for name, score in zip(names, scores):
    print(f"{name} scored {score}")

## 12.2 What does `zip()` return?
`zip(iterable1, iterable2, ...)` combines corresponding items into tuples and returns a zip iterator. By default, it stops when the shortest input iterable is exhausted.

In [ ]:
names = ["Shravan", "Narsing", "Alex", "Anushka"]
scores = [85, 78, 90]

paired = list(zip(names, scores))
print(paired)

In this example, only three pairs are produced because `scores` contains three values. `Anushka` has no corresponding score, so it is not included. No `IndexError` needs to be handled.

## 12.3 Zipping more than two iterables and objects
`zip()` can combine more than two iterables. The class also demonstrates that elements can be objects; `zip()` pairs values and does not require them to be strings or numbers.

In [ ]:
departments = ["QA", "Data", "AI"]
batch_users = [User(3.0), User(3.0), User(4.0)]

for department, user in zip(departments, batch_users):
    print(department, user.batch_number)

If you want to inspect the zipped result as a list of tuples, use `list(zip(...))`. If you only need to loop over the pairs once, loop directly over `zip(...)`.

**Technical note:** Python also supports `zip(..., strict=True)` in modern versions to raise an error when iterable lengths differ. That option was not covered in the supplied class material, so the main examples use the default shortest-iterable behaviour.

# Part 13 — Switch-case idea in Python: `match-case`

## 13.1 `if-elif-else` version
The class starts with a function that chooses an action based on a numeric value.

In [ ]:
def calculation_if(action):
    if action == 1:
        print("Addition selected by user")
    elif action == 2:
        print("Subtraction selected by user")
    else:
        print("Action not defined")

calculation_if(1)
calculation_if(2)
calculation_if(3)

## 13.2 `match-case` version
Python introduced structural pattern matching in Python 3.10. The class describes this as Python's match-case counterpart to the switch-case construct used in some other languages.

In [ ]:
def calculation(action):
    match action:
        case 1:
            print("Match-Case: Addition selected by user")
        case 2:
            print("Match-Case: Subtraction selected by user")
        case _:
            print("Match-Case: Action not defined")

calculation(1)
calculation(2)
calculation(3)

How it works:
1. `match action` specifies the value to examine.
2. Python checks the `case` patterns in order.
3. The first matching case is selected.
4. `case _` is the wildcard pattern and acts as the catch-all case here.

For simple equality choices, this can be a clear alternative to a chain of `if`/`elif` statements. `match-case` can also do more advanced structural pattern matching, but that was not covered in this class.

## 13.3 Optional interactive input
The original notebook used `input()` to ask the user for an action. Run this in a normal Python script or uncomment it in a notebook when you want an interactive prompt.

In [ ]:
# action = int(input("Enter 1 for addition, 2 for subtraction: "))
# calculation(action)

## 13.4 `_` as a wildcard and as an unused variable
The class shows `_` in two contexts:
- In `case _`, it is a wildcard pattern that matches anything not matched earlier.
- In a loop such as `for _ in range(...)`, it is a conventional variable name indicating that the value is not needed.

These uses look similar but have different language roles.

In [ ]:
for _ in range(1, 10):
    print("This loop value is intentionally unused")

for _, value in enumerate(range(1, 5)):
    print(value)

# Part 14 — Combining the tools

## 14.1 Filter, transform and enumerate
This example combines several concepts from the class: `filter()`, `map()`, and `enumerate()`.

In [ ]:
numbers = list(range(1, 11))

even_values = filter(lambda number: number % 2 == 0, numbers)
squared_values = map(lambda number: number ** 2, even_values)

for position, value in enumerate(squared_values, start=1):
    print(position, value)

## 14.2 Zip names with transformed scores

In [ ]:
names = ["Asha", "Ravi", "Meera"]
scores = [85, 72, 94]

result = [
    (name, score, "Pass" if score >= 40 else "Fail")
    for name, score in zip(names, scores)
]

print(result)

# Part 15 — Project planning guidance from the class

The end of the transcript shifts from Python syntax to how students can practice using AI tools. The instructor's main point is that a developer should plan the software before asking an AI tool to generate code.

The HR Management System example follows this sequence:
1. Ask what common functionalities an HR management product typically includes.
2. Choose the specific functionality needed instead of requesting every feature.
3. Define the intended scope, such as employee-level shift/roster planning, leave/absence management, attendance tracking and payroll processing.
4. Ask for a package-based folder structure before asking for implementation.
5. Specify that a single `main.py` should act as the entry point and route user choices to the appropriate module/function.
6. Ask the AI to provide code with the corresponding file names.
7. Create the folders/files, place the generated code in the correct files, run it and inspect the flow.
8. Use Python knowledge to understand, test, debug and request targeted changes.

The instructor's message is not “let AI decide everything.” It is: use your understanding of programming to define what should be built, then use AI to help implement it.

### Example project-planning prompt based on the class discussion

In [ ]:
project_request = '''
I want to build an HR Management System in Python.

First, propose a folder/package structure. Do not generate implementation code yet.

Scope:
- Employee-level shift and roster planning
- Absence and leave management
- Attendance tracking
- Payroll processing

Requirements:
- Use separate modules for the different functional areas.
- Keep one main.py as the application entry point.
- When the user selects a menu option in main.py, route to the appropriate module/function.
- Explain the purpose of each file.
- After I approve the folder structure, provide the code file by file and label each code block with its exact file name.
'''
print(project_request)

# Part 16 — Common mistakes and clarifications

| Topic | Common misunderstanding | Correct understanding |
|---|---|---|
| Class variable | Updating `obj.value` always updates the class value | It may create an instance attribute that shadows the class variable |
| Static method | It receives `self` automatically | It receives no automatic first argument |
| `__init__` | It creates the object | `__new__` creates; `__init__` initializes |
| `__str__` / `__repr__` | They are interchangeable | `__str__` is user-friendly; `__repr__` is developer/debug-oriented |
| `__add__` | It changes the meaning of `+` globally | It defines `+` behaviour for instances of that class |
| Getter/setter | A private-style variable cannot be exposed | Methods/properties can provide controlled access |
| `@property` | The getter must be called like a method | Access it with attribute syntax |
| `map()` | It filters values | It transforms every input item |
| `filter()` | It transforms every value | It keeps values based on the predicate result |
| Starred unpacking | It always returns a tuple | The starred target receives a list |
| Generator | It builds all values immediately | It yields values as requested |
| `zip()` | It errors when list lengths differ | By default, it stops at the shortest iterable |
| `match-case` | `_` is an ordinary exact value match | `case _` is a wildcard catch-all pattern |

# Part 17 — Practice exercises

Try these before checking your own solution.

1. Create a `Course` class with a class variable `platform_name` and an instance variable `student_name`. Update the class variable and observe what two objects see.
2. Create a `Temperature` class with a static method that converts Celsius to Fahrenheit.
3. Create a `Book` class with `__str__` and `__repr__`.
4. Create a `BankBalance` class where `+` returns a new balance object and `==` compares balances.
5. Create a `Playlist` class with `__len__` and `__getitem__`.
6. Create an `Employee` class with a private-style salary attribute and a property setter that rejects negative values.
7. Write a lambda that returns the square of a number.
8. Use `map()` to convert a list of temperatures from Celsius to Fahrenheit.
9. Use `filter()` to retain words longer than five characters.
10. Unpack a list into `first`, `second` and `remaining`.
11. Merge three lists using starred unpacking.
12. Use a list comprehension to create cubes of numbers from 1 to 10.
13. Use a dictionary comprehension to map each number from 1 to 20 to its square.
14. Use `enumerate()` to print a numbered list starting from 1.
15. Write a generator that yields multiples of 3 up to a supplied limit.
16. Use `zip()` to pair product names and prices.
17. Write a `match-case` function for menu options 1, 2, 3 and a default case.
18. Plan a small multi-module application before asking an AI tool to generate code.

# Part 18 — Quick revision sheet

| Concept | Remember |
|---|---|
| Class variable / static variable | Shared class-level value unless shadowed on an instance |
| Instance variable | Belongs to an individual object |
| `@staticmethod` | Method in class namespace with no automatic `self` or `cls` |
| `__new__` | Creates an object |
| `__init__` | Initializes an object |
| `__str__` | Readable string form |
| `__repr__` | Developer-oriented representation |
| `__add__` | Behaviour for `+` |
| `__sub__` | Behaviour for `-` |
| `__eq__` | Behaviour for `==` |
| `__len__` | Behaviour for `len(obj)` |
| `__getitem__` | Behaviour for `obj[index]` |
| Getter | Reads an attribute |
| Setter | Updates an attribute |
| `@property` | Attribute-style getter |
| `@name.setter` | Attribute-style setter |
| `@name.deleter` | Attribute-style deleter |
| Lambda | Small anonymous one-expression function |
| `map()` | Transform items |
| `filter()` | Keep matching items |
| Unpacking | Assign iterable elements to variables |
| `*values` | Collect remaining values / unpack positional values depending on context |
| List comprehension | Concise list construction |
| Dict comprehension | Concise dictionary construction |
| `enumerate()` | Counter plus item |
| Iterator | Supports `__iter__()` and `__next__()` |
| Generator | Iterator produced by a generator function |
| `yield` | Produce a value and pause generator execution |
| `zip()` | Pair corresponding items from multiple iterables |
| `match-case` | Pattern matching, introduced in Python 3.10 |
| `case _` | Wildcard catch-all pattern |

# Source coverage and editorial notes

## Sources used
- `25-10-2026.ipynb` — 139-cell class notebook. It contains the class-variable and static-method examples, special methods, getter/setter/property examples, lambda, map, filter, unpacking, comprehensions, enumerate, generators, zip and match-case.
- `GMT20261025-142925_Recording.transcript.vtt` — class recording transcript. It provides explanation and examples around wallet operator overloading, ATM length/indexing, properties, lazy generators, zip with mismatched list lengths, match-case and project-planning guidance.
- `25-10-2026_notes.pdf` — checked, but it had no extractable text layer; no claims were added from unseen PDF content.

## Clarifications made explicit
- The class notebook calls class variables “static variables”; Python commonly calls them class variables.
- Assigning a class-variable name through an instance can create an instance attribute that shadows the class variable.
- `__new__` creates the object; `__init__` initializes it.
- The `Wallet.__eq__` example includes a type check and returns `NotImplemented` for unrelated types.
- `zip()` stops at the shortest iterable by default, which is why the class's example does not need `try/except` for the unmatched final name.
- `match-case` is Python structural pattern matching, available from Python 3.10. The simple numeric example behaves like a switch-style selection.
- The supplied class notebook had an intentionally incomplete `for name, score in :` cell and an unpacking example with too few targets. These are explained as incomplete/error examples rather than retained as executable cells.
- The original `ATM` spelling `transacrion_history` and several transcript auto-caption spelling errors are normalized to `transaction_history` and standard Python terms.

## Scope boundary
The class closes the Python module and previews future NumPy classes. Async programming, regex details and advanced structural pattern matching are mentioned or explicitly deferred, so they are not taught as completed topics in this notebook.